# Agent的高级用法-ToolStrategy
## 1、ToolStrategy的多种结构化输出方式：schema参数

### 1.1 Pydantic类型

使用CloseAI平台的模型

In [ ]:
from dataclasses import dataclass

from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

# 从.env文件中加载环境变量
load_dotenv(override=True)

CLOSEAI_API_KEY = os.getenv("CLOSEAI_API_KEY")
CLOSEAI_BASE_URL = os.getenv("CLOSEAI_BASE_URL")

model = init_chat_model(
    model="gpt-5.4-mini",
    model_provider="openai",
    api_key=CLOSEAI_API_KEY,
    base_url=CLOSEAI_BASE_URL
)

使用OpenRounter平台的模型

In [1]:
from langchain_openrouter import ChatOpenRouter
from dotenv import load_dotenv
import os

load_dotenv(override=True)

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_API_BASE = os.getenv("OPENROUTER_API_BASE")

model = ChatOpenRouter(
    model="openai/gpt-5.4-mini",
    api_key=OPENROUTER_API_KEY,
    base_url=OPENROUTER_API_BASE,
)

举例1：

In [2]:
from pydantic import BaseModel, Field
from langchain.agents.structured_output import ToolStrategy
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from rich import print as rprint

class ContactInfo(BaseModel):
    """用户的联系方式"""
    name: str = Field(description="用户姓名")
    email: str = Field(description="用户邮箱地址")
    phone: str = Field(description="用户的手机号")


agent = create_agent(
    model=model,
    response_format=ToolStrategy(schema=ContactInfo)
)

response = agent.invoke({
    "messages" : [
        HumanMessage(content="从这段话中抽取结构化信息：小明的邮箱是shkstart@atguigu.com,电话是：13012341234")
    ]
})


rprint(response)

{
    'messages': [
        HumanMessage(
            content='从这段话中抽取结构化信息：小明的邮箱是shkstart@atguigu.com,电话是：13012341234',
            additional_kwargs={},
            response_metadata={},
            id='78df2e5f-57fa-4165-83ed-f620feebf8d2'
        ),
        AIMessage(
            content='',
            additional_kwargs={},
            response_metadata={
                'model_name': 'openai/gpt-5.4-mini-20260317',
                'id': 'gen-1780578874-XLP6zGYAQUjzL4qWkrfs',
                'created': 1780578874,
                'object': 'chat.completion',
                'finish_reason': 'tool_calls',
                'logprobs': None,
                'model_provider': 'openrouter'
            },
            id='lc_run--019e92c5-69f9-7d60-b95a-f5c113ea889d-0',
            tool_calls=[
                {
                    'name': 'ContactInfo',
                    'args': {'name': '小明', 'email': 'shkstart@atguigu.com', 'phone': '13012341234'},
                    'id': 'call_uxu76DKex3Had6zhQy4Vy3tS',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 91,
                'output_tokens': 36,
                'total_tokens': 127,
                'input_token_details': {'cache_read': 0, 'cache_creation': 0},
                'output_token_details': {'reasoning': 0}
            }
        ),
        ToolMessage(
            content="Returning structured response: name='小明' email='shkstart@atguigu.com' phone='13012341234'",
            name='ContactInfo',
            id='cb4a9c34-e5ea-49b7-893d-820e8e77e759',
            tool_call_id='call_uxu76DKex3Had6zhQy4Vy3tS'
        )
    ],
    'structured_response': ContactInfo(name='小明', email='shkstart@atguigu.com', phone='13012341234')
}

举例2：添加工具的调用

In [4]:
from langchain_core.messages import SystemMessage
from pydantic import BaseModel, Field
from typing import Literal
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain.tools import tool
from rich import print as rprint

# 定义工具
@tool(parse_docstring=True)
def search_customer_database(query: str) -> str:
    """
    在客户数据库中搜索信息

    Args:
        query (str): 客户查询字符串，例如 "张三" 或 "李四"

    Returns:
        str: 客户记录字符串，包含客户姓名、等级、最近购买日期和累计消费
    """
    # 模拟数据库查询结果
    if "张三" in query.lower():
        return "客户记录：张三，VIP客户，最近购买日期：2026-01-15，累计消费：$15,000"
    elif "李四" in query.lower():
        return "客户记录：李四，普通客户，最近购买日期：2025-12-20，累计消费：$3,200"
    else:
        return f"关于客户{query}，无记录"


@tool(parse_docstring=True)
def send_email(customer: str) -> str:
    """
    发送感谢邮件

    Args:
        customer (str): 客户名称，例如 "张三" 或 "李四"

    Returns:
        str: 确认消息，包含已发送的客户名称
    """
    return f"已向 {customer} 发送感谢邮件"


# 定义Pydantic Schema
class CustomerAnalysis(BaseModel):
    """客户分析报告"""
    customer_name: str = Field(None, description="客户姓名")
    customer_tier: Literal["潜在客户", "普通客户", "VIP客户", "流失风险"] = Field("潜在客户",description="客户等级,只能是潜在客户、普通客户、VIP客户或流失风险")
    recent_activity: str = Field(None, description="最近活动")
    spending_level: Literal["低", "中", "高"] = Field(None, description="消费水平")
    send_email: bool = Field(False, description="是否已发送感谢邮件")


# 创建智能体
agent = create_agent(
    model=model,
    system_prompt=SystemMessage(content=""
                                        "请分析指定客户的情况："
                                        "1. 先搜索客户数据库了解最新情况 "
                                        "2. 如果是VIP客户，则发送感谢邮件 "
                                        "3. 基于搜索结果生成结构化分析报告 "
                                        "4. 如果用户提问与客户记录无关或找不到客户信息，则返回空对象，不发送感谢邮件"
                                ),
    tools=[search_customer_database, send_email],
    response_format=ToolStrategy(schema=CustomerAnalysis)
)

# 执行分析
result = agent.invoke({
    # "messages": [{"role": "user", "content": "请分析客户张三"}]
    "messages": [{"role": "user","content": "请分析客户李四"}]
    # "messages": [{"role": "user","content": "请分析客户王五"}]
    # "messages": [{"role": "user","content": "今天天气如何"}]
})

# 处理结果
rprint(result)

# if "structured_response" in result:
#     analysis = result["structured_response"]
#     print(analysis)

{
    'messages': [
        HumanMessage(
            content='请分析客户李四',
            additional_kwargs={},
            response_metadata={},
            id='79854a0a-9707-4575-95c7-77ac32a50f73'
        ),
        AIMessage(
            content='',
            additional_kwargs={},
            response_metadata={
                'model_name': 'openai/gpt-5.4-mini-20260317',
                'id': 'gen-1780579287-E1I5IxSKVyinDhRgAfAL',
                'created': 1780579287,
                'object': 'chat.completion',
                'finish_reason': 'tool_calls',
                'logprobs': None,
                'model_provider': 'openrouter'
            },
            id='lc_run--019e92cb-b764-7fc3-85bc-e6a2eee48579-0',
            tool_calls=[
                {
                    'name': 'search_customer_database',
                    'args': {'query': '李四'},
                    'id': 'call_BTGjiFz9JCFzlB0gjIND47Xq',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 287,
                'output_tokens': 20,
                'total_tokens': 307,
                'input_token_details': {'cache_read': 0, 'cache_creation': 0},
                'output_token_details': {'reasoning': 0}
            }
        ),
        ToolMessage(
            content='客户记录：李四，普通客户，最近购买日期：2025-12-20，累计消费：$3,200',
            name='search_customer_database',
            id='8740e15b-35fb-425e-87cf-781daf67c38f',
            tool_call_id='call_BTGjiFz9JCFzlB0gjIND47Xq'
        ),
        AIMessage(
            content='',
            additional_kwargs={},
            response_metadata={
                'model_name': 'openai/gpt-5.4-mini-20260317',
                'id': 'gen-1780579288-WwrrYsj9bS5xUzE58wXT',
                'created': 1780579288,
                'object': 'chat.completion',
                'finish_reason': 'tool_calls',
                'logprobs': None,
                'model_provider': 'openrouter'
            },
            id='lc_run--019e92cb-be74-7370-a488-a1b96830d9ac-0',
            tool_calls=[
                {
                    'name': 'CustomerAnalysis',
                    'args': {
                        'customer_name': '李四',
                        'customer_tier': '普通客户',
                        'recent_activity': '最近购买日期：2025-12-20',
                        'spending_level': '中',
                        'send_email': False
                    },
                    'id': 'call_Cs2Z94ITkeHSJikMnMRNY5xY',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 346,
                'output_tokens': 52,
                'total_tokens': 398,
                'input_token_details': {'cache_read': 0, 'cache_creation': 0},
                'output_token_details': {'reasoning': 0}
            }
        ),
        ToolMessage(
            content="Returning structured response: customer_name='李四' customer_tier='普通客户' 
recent_activity='最近购买日期：2025-12-20' spending_level='中' send_email=False",
            name='CustomerAnalysis',
            id='8c4a51cf-8634-4f1e-b212-994df2fcc6ea',
            tool_call_id='call_Cs2Z94ITkeHSJikMnMRNY5xY'
        )
    ],
    'structured_response': CustomerAnalysis(
        customer_name='李四',
        customer_tier='普通客户',
        recent_activity='最近购买日期：2025-12-20',
        spending_level='中',
        send_email=False
    )
}

### 1.2 TypedDict类型

举例1：

In [5]:
from typing import TypedDict, Annotated
from pydantic import BaseModel, Field
from langchain.agents.structured_output import ToolStrategy
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from rich import print as rprint

class ContactInfo(TypedDict):
    """用户的联系方式"""
    name: Annotated[str, ..., "用户姓名"]
    email: Annotated[str, ..., "用户邮箱地址"]
    phone: Annotated[str, ..., "用户的手机号"]

agent = create_agent(
    model=model,
    response_format=ToolStrategy(schema=ContactInfo)
)

response = agent.invoke({
    "messages" : [
        HumanMessage(content="从这段话中抽取结构化信息：小明的邮箱是shkstart@atguigu.com,电话是：13012341234")
    ]
})


rprint(response)

{
    'messages': [
        HumanMessage(
            content='从这段话中抽取结构化信息：小明的邮箱是shkstart@atguigu.com,电话是：13012341234',
            additional_kwargs={},
            response_metadata={},
            id='55ae09ec-0bd7-4003-93d2-7181f6125f90'
        ),
        AIMessage(
            content='',
            additional_kwargs={},
            response_metadata={
                'model_name': 'openai/gpt-5.4-mini-20260317',
                'id': 'gen-1780579423-XolcKSVsB7BeRDw9pgJi',
                'created': 1780579423,
                'object': 'chat.completion',
                'finish_reason': 'tool_calls',
                'logprobs': None,
                'model_provider': 'openrouter'
            },
            id='lc_run--019e92cd-cc9d-7923-a8e2-6f08c996e40c-0',
            tool_calls=[
                {
                    'name': 'ContactInfo',
                    'args': {'name': '小明', 'email': 'shkstart@atguigu.com', 'phone': '13012341234'},
                    'id': 'call_t90greoA1vHX1gHbnvdV7sHu',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 79,
                'output_tokens': 36,
                'total_tokens': 115,
                'input_token_details': {'cache_read': 0, 'cache_creation': 0},
                'output_token_details': {'reasoning': 0}
            }
        ),
        ToolMessage(
            content="Returning structured response: {'name': '小明', 'email': 'shkstart@atguigu.com', 'phone': 
'13012341234'}",
            name='ContactInfo',
            id='f08b2704-6f0d-4b7a-9945-173254ca8d63',
            tool_call_id='call_t90greoA1vHX1gHbnvdV7sHu'
        )
    ],
    'structured_response': {'name': '小明', 'email': 'shkstart@atguigu.com', 'phone': '13012341234'}
}

举例2：

In [ ]:
from langchain_core.messages import SystemMessage
from pydantic import BaseModel, Field
from typing import Literal, Optional
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain.tools import tool
from rich import print as rprint

# 定义工具
@tool(parse_docstring=True)
def search_customer_database(query: str) -> str:
    """
    在客户数据库中搜索信息

    Args:
        query (str): 客户查询字符串，例如 "张三" 或 "李四"

    Returns:
        str: 客户记录字符串，包含客户姓名、等级、最近购买日期和累计消费
    """
    # 模拟数据库查询结果
    if "张三" in query.lower():
        return "客户记录：张三，VIP客户，最近购买日期：2026-01-15，累计消费：$15,000"
    elif "李四" in query.lower():
        return "客户记录：李四，普通客户，最近购买日期：2025-12-20，累计消费：$3,200"
    else:
        return f"关于客户{query}，无记录"


@tool(parse_docstring=True)
def send_email(customer: str) -> str:
    """
    发送感谢邮件

    Args:
        customer (str): 客户名称，例如 "张三" 或 "李四"

    Returns:
        str: 确认消息，包含已发送的客户名称
    """
    return f"已向 {customer} 发送感谢邮件"


# 使用 TypedDict 定义客户分析报告 Schema
class CustomerAnalysis(TypedDict):
    """客户分析报告"""
    customer_name: Annotated[Optional[str], None, "客户姓名"]
    customer_tier: Annotated[Literal["潜在客户", "普通客户", "VIP客户", "流失风险"], "潜在客户", "客户等级"]
    recent_activity: Annotated[Optional[str], None, "最近活动"]
    spending_level: Annotated[Optional[Literal["低", "中", "高"]], None, "消费水平"]
    send_email: Annotated[bool, False, "是否已发送感谢邮件"]

# 创建智能体
agent = create_agent(
    model=model,
    system_prompt=SystemMessage(content=""
                                        "请分析指定客户的情况："
                                        "1. 先搜索客户数据库了解最新情况 "
                                        "2. 如果是VIP客户，则发送感谢邮件 "
                                        "3. 基于搜索结果生成结构化分析报告 "
                                        "4. 如果用户提问与客户记录无关或找不到客户信息，则返回空对象，不发送感谢邮件"
                                ),
    tools=[search_customer_database, send_email],
    response_format=ToolStrategy(schema=CustomerAnalysis)
)

# 执行分析
result = agent.invoke({
    # "messages": [{"role": "user", "content": "请分析客户张三"}]
    "messages": [{"role": "user","content": "请分析客户李四"}]
    # "messages": [{"role": "user","content": "请分析客户王五"}]
    # "messages": [{"role": "user","content": "今天天气如何"}]
})

# 处理结果
rprint(result)

# if "structured_response" in result:
#     analysis = result["structured_response"]
#     print(analysis)

### 1.3 JsonSchema类型

举例1：

In [7]:
from typing import TypedDict, Annotated
from pydantic import BaseModel, Field
from langchain.agents.structured_output import ToolStrategy
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from rich import print as rprint

json_schema = {
    "title": "ContactInfo",
    "description": "用户的联系方式",
    "type": "object",
    "properties": {
        "name": {
            "description": "用户姓名",
            "type": "string"
        },
        "email": {
            "description": "用户邮箱地址",
            "type": "string"
        },
        "phone": {
            "description": "用户的手机号",
            "type": "string"
        }
    },
    "required": [
        "name",
        "email",
        "phone"
    ]
}

agent = create_agent(
    model=model,
    response_format=ToolStrategy(schema=json_schema)
)

response = agent.invoke({
    "messages" : [
        HumanMessage(content="从这段话中抽取结构化信息：小明的邮箱是shkstart@atguigu.com,电话是：13012341234")
    ]
})


rprint(response)

{
    'messages': [
        HumanMessage(
            content='从这段话中抽取结构化信息：小明的邮箱是shkstart@atguigu.com,电话是：13012341234',
            additional_kwargs={},
            response_metadata={},
            id='6e33636c-cefc-4563-be20-8ebc35e68769'
        ),
        AIMessage(
            content='',
            additional_kwargs={},
            response_metadata={
                'model_name': 'openai/gpt-5.4-mini-20260317',
                'id': 'gen-1780579620-azPHd0RZQccqBKm5SeL5',
                'created': 1780579620,
                'object': 'chat.completion',
                'finish_reason': 'tool_calls',
                'logprobs': None,
                'model_provider': 'openrouter'
            },
            id='lc_run--019e92d0-cc8d-7d73-90e0-4deb355dac81-0',
            tool_calls=[
                {
                    'name': 'ContactInfo',
                    'args': {'name': '小明', 'email': 'shkstart@atguigu.com', 'phone': '13012341234'},
                    'id': 'call_RUzZCkBMn1IH3y6UkJIm1Nkw',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 91,
                'output_tokens': 36,
                'total_tokens': 127,
                'input_token_details': {'cache_read': 0, 'cache_creation': 0},
                'output_token_details': {'reasoning': 0}
            }
        ),
        ToolMessage(
            content="Returning structured response: {'name': '小明', 'email': 'shkstart@atguigu.com', 'phone': 
'13012341234'}",
            name='ContactInfo',
            id='7ff74238-87f2-42e7-8a13-c30735c3648b',
            tool_call_id='call_RUzZCkBMn1IH3y6UkJIm1Nkw'
        )
    ],
    'structured_response': {'name': '小明', 'email': 'shkstart@atguigu.com', 'phone': '13012341234'}
}

举例2：

In [8]:
from langchain_core.messages import SystemMessage
from pydantic import BaseModel, Field
from typing import Literal, Optional
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain.tools import tool
from rich import print as rprint

# 定义工具
@tool(parse_docstring=True)
def search_customer_database(query: str) -> str:
    """
    在客户数据库中搜索信息

    Args:
        query (str): 客户查询字符串，例如 "张三" 或 "李四"

    Returns:
        str: 客户记录字符串，包含客户姓名、等级、最近购买日期和累计消费
    """
    # 模拟数据库查询结果
    if "张三" in query.lower():
        return "客户记录：张三，VIP客户，最近购买日期：2026-01-15，累计消费：$15,000"
    elif "李四" in query.lower():
        return "客户记录：李四，普通客户，最近购买日期：2025-12-20，累计消费：$3,200"
    else:
        return f"关于客户{query}，无记录"


@tool(parse_docstring=True)
def send_email(customer: str) -> str:
    """
    发送感谢邮件

    Args:
        customer (str): 客户名称，例如 "张三" 或 "李四"

    Returns:
        str: 确认消息，包含已发送的客户名称
    """
    return f"已向 {customer} 发送感谢邮件"


# 使用 json_schema 定义客户分析报告 Schema
customer_analysis_schema = {
    "title": "CustomerAnalysis",
    "type": "object",
    "description": "客户分析报告",
    "properties": {
        "customer_name": {
            "type": "string",
            "default": "",
            "description": "客户姓名"
        },
        "customer_tier": {
            "type": "string",
            "enum": ["潜在客户", "普通客户", "VIP客户", "流失风险"],
            "default": "潜在客户",
            "description": "客户等级"
        },
        "recent_activity": {
            "type": "string",
            "default": "",
            "description": "最近活动"
        },
        "spending_level": {
            "type": "string",
            "enum": ["低", "中", "高"],
            "default": "低",
            "description": "消费水平"
        },
        "send_email": {
            "type": "boolean",
            "default": False,
            "description": "是否已发送感谢邮件"
        }
    },
    # 所有字段都是必须输出的
    "required": ["customer_name", "customer_tier", "recent_activity", "spending_level"]
}

# 创建智能体
agent = create_agent(
    model=model,
    system_prompt=SystemMessage(content=""
                                        "请分析指定客户的情况："
                                        "1. 先搜索客户数据库了解最新情况 "
                                        "2. 如果是VIP客户，则发送感谢邮件 "
                                        "3. 基于搜索结果生成结构化分析报告 "
                                        "4. 如果用户提问与客户记录无关或找不到客户信息，则返回空对象，不发送感谢邮件"
                                ),
    tools=[search_customer_database, send_email],
    response_format=ToolStrategy(schema=customer_analysis_schema)
)

# 执行分析
result = agent.invoke({
    "messages": [{"role": "user", "content": "请分析客户张三"}]
    # "messages": [{"role": "user","content": "请分析客户李四"}]
    # "messages": [{"role": "user","content": "请分析客户王五"}]
    # "messages": [{"role": "user","content": "今天天气如何"}]
})

# 处理结果
rprint(result)

# if "structured_response" in result:
#     analysis = result["structured_response"]
#     print(analysis)

{
    'messages': [
        HumanMessage(
            content='请分析客户张三',
            additional_kwargs={},
            response_metadata={},
            id='8af0add2-8ffe-4edb-b748-16cae78efd5f'
        ),
        AIMessage(
            content='',
            additional_kwargs={},
            response_metadata={
                'model_name': 'openai/gpt-5.4-mini-20260317',
                'id': 'gen-1780579723-el5E27AxUneGmVOFiP9r',
                'created': 1780579723,
                'object': 'chat.completion',
                'finish_reason': 'tool_calls',
                'logprobs': None,
                'model_provider': 'openrouter'
            },
            id='lc_run--019e92d2-6001-7652-84bb-c95876ceaf7c-0',
            tool_calls=[
                {
                    'name': 'search_customer_database',
                    'args': {'query': '张三'},
                    'id': 'call_ziqtuGPsERJe041EILNorU4t',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 287,
                'output_tokens': 20,
                'total_tokens': 307,
                'input_token_details': {'cache_read': 0, 'cache_creation': 0},
                'output_token_details': {'reasoning': 0}
            }
        ),
        ToolMessage(
            content='客户记录：张三，VIP客户，最近购买日期：2026-01-15，累计消费：$15,000',
            name='search_customer_database',
            id='8e610101-de9b-4645-9d90-17d0e7747dbb',
            tool_call_id='call_ziqtuGPsERJe041EILNorU4t'
        ),
        AIMessage(
            content='',
            additional_kwargs={},
            response_metadata={
                'model_name': 'openai/gpt-5.4-mini-20260317',
                'id': 'gen-1780579724-qvadmoOYoYhFxfiDYwKA',
                'created': 1780579724,
                'object': 'chat.completion',
                'finish_reason': 'tool_calls',
                'logprobs': None,
                'model_provider': 'openrouter'
            },
            id='lc_run--019e92d2-6780-76d0-a2ff-8cdd1f9c0031-0',
            tool_calls=[
                {
                    'name': 'send_email',
                    'args': {'customer': '张三'},
                    'id': 'call_3Uhoe0h6uG5mJfsSTggzDsJa',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 346,
                'output_tokens': 19,
                'total_tokens': 365,
                'input_token_details': {'cache_read': 0, 'cache_creation': 0},
                'output_token_details': {'reasoning': 0}
            }
        ),
        ToolMessage(
            content='已向 张三 发送感谢邮件',
            name='send_email',
            id='95c8201a-c1a1-42ad-94f8-13148f922258',
            tool_call_id='call_3Uhoe0h6uG5mJfsSTggzDsJa'
        ),
        AIMessage(
            content='',
            additional_kwargs={},
            response_metadata={
                'model_name': 'openai/gpt-5.4-mini-20260317',
                'id': 'gen-1780579725-O3EKTrZ9C8d6eDzF9jSH',
                'created': 1780579725,
                'object': 'chat.completion',
                'finish_reason': 'tool_calls',
                'logprobs': None,
                'model_provider': 'openrouter'
            },
            id='lc_run--019e92d2-6b97-78c1-8af3-a6f1dd571cb2-0',
            tool_calls=[
                {
                    'name': 'CustomerAnalysis',
                    'args': {
                        'customer_name': '张三',
                        'customer_tier': 'VIP客户',
                        'recent_activity': '最近购买日期：2026-01-15，累计消费：$15,000',
                        'spending_level': '高',
                        'send_email': True
                    },
                    'id': 'call_3ET21FVLEko1PSbc6OCkXMKz',
                    'type': 'tool_call'
                }
            ],
 

### 1.4 @dataclass类型

举例1：

In [10]:
from pydantic import BaseModel, Field
from langchain.agents.structured_output import ToolStrategy
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from rich import print as rprint
from dataclasses import dataclass

@dataclass
class ContactInfo:
    """用户的联系方式"""
    name: str = Field(description="用户姓名")
    email: str = Field(description="用户邮箱地址")
    phone: str = Field(description="用户的手机号")


agent = create_agent(
    model=model,
    response_format=ToolStrategy(schema=ContactInfo)
)

response = agent.invoke({
    "messages" : [
        HumanMessage(content="从这段话中抽取结构化信息：小明的邮箱是shkstart@atguigu.com,电话是：13012341234")
    ]
})


rprint(response)

{
    'messages': [
        HumanMessage(
            content='从这段话中抽取结构化信息：小明的邮箱是shkstart@atguigu.com,电话是：13012341234',
            additional_kwargs={},
            response_metadata={},
            id='134c4115-d140-42a7-b903-a89dcbe9d72a'
        ),
        AIMessage(
            content='',
            additional_kwargs={},
            response_metadata={
                'model_name': 'openai/gpt-5.4-mini-20260317',
                'id': 'gen-1780579818-MFFyQbgZ4SkN8y2RFRU9',
                'created': 1780579818,
                'object': 'chat.completion',
                'finish_reason': 'tool_calls',
                'logprobs': None,
                'model_provider': 'openrouter'
            },
            id='lc_run--019e92d3-d033-7fa0-bcc3-10ff3a44da7b-0',
            tool_calls=[
                {
                    'name': 'ContactInfo',
                    'args': {'name': '小明', 'email': 'shkstart@atguigu.com', 'phone': '13012341234'},
                    'id': 'call_BrVGldDqTvtwJDRQxwNeI5wj',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 91,
                'output_tokens': 36,
                'total_tokens': 127,
                'input_token_details': {'cache_read': 0, 'cache_creation': 0},
                'output_token_details': {'reasoning': 0}
            }
        ),
        ToolMessage(
            content="Returning structured response: ContactInfo(name='小明', email='shkstart@atguigu.com', 
phone='13012341234')",
            name='ContactInfo',
            id='84b976da-8c42-486c-b6da-e7b892a6a641',
            tool_call_id='call_BrVGldDqTvtwJDRQxwNeI5wj'
        )
    ],
    'structured_response': ContactInfo(name='小明', email='shkstart@atguigu.com', phone='13012341234')
}

举例2：

In [ ]:
from langchain_core.messages import SystemMessage
from pydantic import BaseModel, Field
from typing import Literal
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain.tools import tool
from rich import print as rprint

# 定义工具
@tool(parse_docstring=True)
def search_customer_database(query: str) -> str:
    """
    在客户数据库中搜索信息

    Args:
        query (str): 客户查询字符串，例如 "张三" 或 "李四"

    Returns:
        str: 客户记录字符串，包含客户姓名、等级、最近购买日期和累计消费
    """
    # 模拟数据库查询结果
    if "张三" in query.lower():
        return "客户记录：张三，VIP客户，最近购买日期：2026-01-15，累计消费：$15,000"
    elif "李四" in query.lower():
        return "客户记录：李四，普通客户，最近购买日期：2025-12-20，累计消费：$3,200"
    else:
        return f"关于客户{query}，无记录"


@tool(parse_docstring=True)
def send_email(customer: str) -> str:
    """
    发送感谢邮件

    Args:
        customer (str): 客户名称，例如 "张三" 或 "李四"

    Returns:
        str: 确认消息，包含已发送的客户名称
    """
    return f"已向 {customer} 发送感谢邮件"


# 定义dataclass
@dataclass
class CustomerAnalysis:
    """客户分析报告"""
    customer_name: str = Field(None, description="客户姓名")
    customer_tier: Literal["潜在客户", "普通客户", "VIP客户", "流失风险"] = Field("潜在客户",description="客户等级,只能是潜在客户、普通客户、VIP客户或流失风险")
    recent_activity: str = Field(None, description="最近活动")
    spending_level: Literal["低", "中", "高"] = Field(None, description="消费水平")
    send_email: bool = Field(False, description="是否已发送感谢邮件")


# 创建智能体
agent = create_agent(
    model=model,
    system_prompt=SystemMessage(content=""
                                        "请分析指定客户的情况："
                                        "1. 先搜索客户数据库了解最新情况 "
                                        "2. 如果是VIP客户，则发送感谢邮件 "
                                        "3. 基于搜索结果生成结构化分析报告 "
                                        "4. 如果用户提问与客户记录无关或找不到客户信息，则返回空对象，不发送感谢邮件"
                                ),
    tools=[search_customer_database, send_email],
    response_format=ToolStrategy(schema=CustomerAnalysis)
)

# 执行分析
result = agent.invoke({
    # "messages": [{"role": "user", "content": "请分析客户张三"}]
    "messages": [{"role": "user","content": "请分析客户李四"}]
    # "messages": [{"role": "user","content": "请分析客户王五"}]
    # "messages": [{"role": "user","content": "今天天气如何"}]
})

# 处理结果
rprint(result)

# if "structured_response" in result:
#     analysis = result["structured_response"]
#     print(analysis)

### 1.5 多schema联合模式

举例

In [12]:
from pydantic import BaseModel, Field
from typing import Union
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain.messages import HumanMessage


class ContactInfo(BaseModel):
    """用户的联系方式"""
    name: str = Field(description="用户姓名")
    email: str = Field(description="用户邮箱地址")
    phone: str = Field(description="用户的手机号")

class EventInfo(BaseModel):
    """事件详情"""
    event_name: str = Field(description="事件名称")
    date: str = Field(description="事件发生日期")

agent = create_agent(
    model=model,
    response_format=ToolStrategy(schema=Union[ContactInfo, EventInfo])
)

response = agent.invoke(
    {
        "messages": [
            HumanMessage("从这段话中抽取结构化信息：小明的邮箱地址为：shkstart@atguigu.com，手机号：12345678912")
        ]
    }
)

for msg in response["messages"]:
    msg.pretty_print()

# print(response["structured_response"])

================================ Human Message =================================

从这段话中抽取结构化信息：小明的邮箱地址为：shkstart@atguigu.com，手机号：12345678912
================================== Ai Message ==================================
Tool Calls:
  ContactInfo (call_6YjPl6OJq5LwvKjZwdKbK5HM)
 Call ID: call_6YjPl6OJq5LwvKjZwdKbK5HM
  Args:
    name: 小明
    email: shkstart@atguigu.com
    phone: 12345678912
================================= Tool Message =================================
Name: ContactInfo

Returning structured response: name='小明' email='shkstart@atguigu.com' phone='12345678912'


In [13]:
response = agent.invoke(
    {
        "messages": [
            HumanMessage("从这段话中抽取结构化信息：2026年高考报名人数突破1200万")
        ]
    }
)

for msg in response["messages"]:
    msg.pretty_print()

print(response["structured_response"])

================================ Human Message =================================

从这段话中抽取结构化信息：2026年高考报名人数突破1200万
================================== Ai Message ==================================
Tool Calls:
  EventInfo (call_M2mVescZfCkhS6DHQAy3WxBW)
 Call ID: call_M2mVescZfCkhS6DHQAy3WxBW
  Args:
    event_name: 高考报名人数突破1200万
    date: 2026年
================================= Tool Message =================================
Name: EventInfo

Returning structured response: event_name='高考报名人数突破1200万' date='2026年'
event_name='高考报名人数突破1200万' date='2026年'


## 2、自定义工具消息：tool_message_content参数

举例：

In [14]:
from pydantic import BaseModel, Field
from langchain.agents.structured_output import ToolStrategy
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from rich import print as rprint

class ContactInfo(BaseModel):
    """用户的联系方式"""
    name: str = Field(description="用户姓名")
    email: str = Field(description="用户邮箱地址")
    phone: str = Field(description="用户的手机号")


agent = create_agent(
    model=model,
    response_format=ToolStrategy(ContactInfo)
)

response = agent.invoke({
        "messages": [
            HumanMessage("从这段话中抽取结构化信息：小明的邮箱地址为：songhk@atguigu.com，手机号：12345678912")
        ]
    }
)

rprint(response)

{
    'messages': [
        HumanMessage(
            content='从这段话中抽取结构化信息：小明的邮箱地址为：songhk@atguigu.com，手机号：12345678912',
            additional_kwargs={},
            response_metadata={},
            id='a6c7fa5e-8493-4eac-b714-deef07a5fbd7'
        ),
        AIMessage(
            content='',
            additional_kwargs={},
            response_metadata={
                'model_name': 'openai/gpt-5.4-mini-20260317',
                'id': 'gen-1780583509-okEjng1PUepn9hSS7zID',
                'created': 1780583509,
                'object': 'chat.completion',
                'finish_reason': 'tool_calls',
                'logprobs': None,
                'model_provider': 'openrouter'
            },
            id='lc_run--019e930c-22c9-79c1-8dc7-5ca8249f5422-0',
            tool_calls=[
                {
                    'name': 'ContactInfo',
                    'args': {'name': '小明', 'email': 'songhk@atguigu.com', 'phone': '12345678912'},
                    'id': 'call_qdjCgRH5xh8IVsPRKG5DrDKa',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 91,
                'output_tokens': 35,
                'total_tokens': 126,
                'input_token_details': {'cache_read': 0, 'cache_creation': 0},
                'output_token_details': {'reasoning': 0}
            }
        ),
        ToolMessage(
            content="Returning structured response: name='小明' email='songhk@atguigu.com' phone='12345678912'",
            name='ContactInfo',
            id='6e1b976c-1e86-41ba-baf0-7a220b95fb83',
            tool_call_id='call_qdjCgRH5xh8IVsPRKG5DrDKa'
        )
    ],
    'structured_response': ContactInfo(name='小明', email='songhk@atguigu.com', phone='12345678912')
}

作为对比：

In [15]:
from pydantic import BaseModel, Field
from langchain.agents.structured_output import ToolStrategy
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from rich import print as rprint

class ContactInfo(BaseModel):
    """用户的联系方式"""
    name: str = Field(description="用户姓名")
    email: str = Field(description="用户邮箱地址")
    phone: str = Field(description="用户的手机号")


agent = create_agent(
    model=model,
    response_format=ToolStrategy(
        schema = ContactInfo,
        tool_message_content="已成功抽取信息"
    )
)

response = agent.invoke({
        "messages": [
            HumanMessage("从这段话中抽取结构化信息：小明的邮箱地址为：songhk@atguigu.com，手机号：12345678912")
        ]
    }
)

rprint(response)

{
    'messages': [
        HumanMessage(
            content='从这段话中抽取结构化信息：小明的邮箱地址为：songhk@atguigu.com，手机号：12345678912',
            additional_kwargs={},
            response_metadata={},
            id='e9f9e5cf-1756-41d0-9fe1-9890f3f7d5c2'
        ),
        AIMessage(
            content='',
            additional_kwargs={},
            response_metadata={
                'model_name': 'openai/gpt-5.4-mini-20260317',
                'id': 'gen-1780583576-WUxj4xen4iAK3jjBHQ9w',
                'created': 1780583576,
                'object': 'chat.completion',
                'finish_reason': 'tool_calls',
                'logprobs': None,
                'model_provider': 'openrouter'
            },
            id='lc_run--019e930d-2831-7312-b447-523772142711-0',
            tool_calls=[
                {
                    'name': 'ContactInfo',
                    'args': {'name': '小明', 'email': 'songhk@atguigu.com', 'phone': '12345678912'},
                    'id': 'call_all5ItV4o90OOe8lLovqqLxu',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 91,
                'output_tokens': 35,
                'total_tokens': 126,
                'input_token_details': {'cache_read': 0, 'cache_creation': 0},
                'output_token_details': {'reasoning': 0}
            }
        ),
        ToolMessage(
            content='已成功抽取信息',
            name='ContactInfo',
            id='e57c0a54-dd0e-4943-a99b-67e6fd8a1af0',
            tool_call_id='call_all5ItV4o90OOe8lLovqqLxu'
        )
    ],
    'structured_response': ContactInfo(name='小明', email='songhk@atguigu.com', phone='12345678912')
}